# 1. Collect Dataset

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from pathlib import Path
import time

# -------------------------
# 1. Configuration
# -------------------------
SEQUENCE_LENGTH = 30
CAMERA_ID = 0

DATA_DIR = Path("dataset")
CLASSES = ["correct", "incorrect"]

for class_name in CLASSES:
    (DATA_DIR / class_name).mkdir(
        parents=True,
        exist_ok=True
    )

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils


# -------------------------
# 2. Extract 132 features
# -------------------------
def extract_landmarks(results):
    if results.pose_landmarks is None:
        return None

    landmarks = results.pose_landmarks.landmark

    # Shoulder landmarks
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]

    center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2

    scale = np.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2
        + (left_shoulder.y - right_shoulder.y) ** 2
    )

    scale = max(scale, 1e-6)

    values = []

    for landmark in landmarks:
        # Normalize coordinates relative to shoulders
        x = (landmark.x - center_x) / scale
        y = (landmark.y - center_y) / scale
        z = landmark.z / scale
        visibility = landmark.visibility

        values.extend([x, y, z, visibility])

    return np.array(values, dtype=np.float32)


# -------------------------
# 3. Camera setup
# -------------------------
cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError(
        "เปิดกล้องไม่ได้ กรุณาตรวจสอบ CAMERA_ID"
    )

with mp_pose.Pose(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    while True:
        success, frame = cap.read()

        if not success:
            print("อ่านภาพจากกล้องไม่ได้")
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        results = pose.process(rgb)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

        correct_count = len(
            list((DATA_DIR / "correct").glob("*.npy"))
        )
        incorrect_count = len(
            list((DATA_DIR / "incorrect").glob("*.npy"))
        )

        cv2.putText(
            frame,
            f"Correct: {correct_count}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (0, 255, 0), 2
        )

        cv2.putText(
            frame,
            f"Incorrect: {incorrect_count}",
            (20, 65),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (0, 165, 255), 2
        )

        cv2.putText(
            frame,
            "C: Capture | Q: Quit",
            (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (255, 255, 255), 2
        )

        cv2.imshow("SaluteAI - Data Collection", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

        elif key == ord("c"):
            print("\nChoose data class:")
            print("1 = Correct salute")
            print("2 = Incorrect salute")

            choice = input(
                "Enter 1 or 2: "
            ).strip()

            if choice not in ["1", "2"]:
                print("Invalid choice")
                continue

            class_name = (
                "correct" if choice == "1"
                else "incorrect"
            )

            sequence = []
            print(
                f"Get ready: recording {class_name}..."
            )

            # Give user time to prepare
            for remaining in [3, 2, 1]:
                countdown_start = time.time()

                while time.time() - countdown_start < 1:
                    ret, countdown_frame = cap.read()

                    if not ret:
                        break

                    countdown_frame = cv2.flip(
                        countdown_frame, 1
                    )

                    cv2.putText(
                        countdown_frame,
                        str(remaining),
                        (250, 200),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        3, (0, 255, 255), 5
                    )

                    cv2.imshow(
                        "SaluteAI - Data Collection",
                        countdown_frame
                    )
                    cv2.waitKey(1)

            # Collect 30 valid frames
            while len(sequence) < SEQUENCE_LENGTH:
                ret, capture_frame = cap.read()

                if not ret:
                    continue

                capture_frame = cv2.flip(
                    capture_frame, 1
                )

                rgb = cv2.cvtColor(
                    capture_frame,
                    cv2.COLOR_BGR2RGB
                )

                results = pose.process(rgb)

                features = extract_landmarks(results)

                if features is None:
                    cv2.putText(
                        capture_frame,
                        "Body not detected - stand in frame",
                        (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0, 0, 255), 2
                    )
                else:
                    sequence.append(features)

                cv2.putText(
                    capture_frame,
                    f"Frames: {len(sequence)}/{SEQUENCE_LENGTH}",
                    (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 255), 2
                )

                cv2.imshow(
                    "SaluteAI - Data Collection",
                    capture_frame
                )

                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break

            if len(sequence) == SEQUENCE_LENGTH:
                class_dir = DATA_DIR / class_name

                existing = list(
                    class_dir.glob("*.npy")
                )

                file_number = len(existing) + 1

                save_path = class_dir / (
                    f"{class_name}_{file_number:03d}.npy"
                )

                np.save(
                    save_path,
                    np.array(sequence, dtype=np.float32)
                )

                print("Saved:", save_path)
                print(
                    "Shape:",
                    np.array(sequence).shape
                )

cap.release()
cv2.destroyAllWindows()


# 2. สร้าง datasets ที่ถูกและผิด

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

DATA_DIR = Path("dataset")
OUTPUT_DIR = Path("prepared")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)


def load_class(class_name):
    files = sorted(
        (DATA_DIR / class_name).glob("*.npy")
    )

    sequences = []

    for file in files:
        sequence = np.load(file).astype(np.float32)

        if sequence.shape != (30, 132):
            print("Skipping invalid shape:", file, sequence.shape)
            continue

        if not np.isfinite(sequence).all():
            print("Skipping invalid values:", file)
            continue

        sequences.append(sequence)

    if len(sequences) < 5:
        raise ValueError(
            f"Need at least 5 valid sequences in {class_name}. "
            "Collect more data first."
        )

    return np.stack(sequences)


correct = load_class("correct")
incorrect = load_class("incorrect")

print("Correct:", correct.shape)
print("Incorrect:", incorrect.shape)


# Split each class separately before creating pairs
def split_sequences(data):
    train, temp = train_test_split(
        data,
        test_size=0.30,
        random_state=SEED
    )

    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=SEED
    )

    return train, val, test


correct_train, correct_temp_val, correct_temp_test = split_sequences(correct)
incorrect_train, incorrect_temp_val, incorrect_temp_test = split_sequences(incorrect)


# Create balanced positive and negative pairs
def make_pairs(correct_data, incorrect_data):
    if len(correct_data) < 2 or len(incorrect_data) < 2:
        raise ValueError("Each class needs at least 2 sequences per split.")

    pair_a = []
    pair_b = []
    labels = []

    number_of_pairs = 2 * max(
        len(correct_data),
        len(incorrect_data)
    )

    for _ in range(number_of_pairs):
        # Positive pair: two correct examples
        i, j = rng.choice(
            len(correct_data), size=2, replace=False
        )

        pair_a.append(correct_data[i])
        pair_b.append(correct_data[j])
        labels.append(1.0)

        # Negative pair: one correct and one incorrect
        i = rng.integers(len(correct_data))
        j = rng.integers(len(incorrect_data))

        pair_a.append(correct_data[i])
        pair_b.append(incorrect_data[j])
        labels.append(0.0)

    pair_a = np.stack(pair_a).astype(np.float32)
    pair_b = np.stack(pair_b).astype(np.float32)
    labels = np.array(labels, dtype=np.float32)

    # Shuffle the pairs
    order = rng.permutation(len(labels))

    return pair_a[order], pair_b[order], labels[order]


splits = {
    "train": (correct_train, incorrect_train),
    "val": (correct_temp_val, incorrect_temp_val),
    "test": (correct_temp_test, incorrect_temp_test)
}

for split_name, (c_data, i_data) in splits.items():
    a, b, y = make_pairs(c_data, i_data)

    output_path = OUTPUT_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        pair_a=a,
        pair_b=b,
        labels=y
    )

    print(
        f"{split_name}:",
        a.shape,
        b.shape,
        y.shape,
        "saved to", output_path
    )


# 3. Build Siamese

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

Path("models").mkdir(exist_ok=True)

SEED = 42
tf.keras.utils.set_random_seed(SEED)

SEQUENCE_LENGTH = 30
FEATURES = 132


# -------------------------
# 1. Load prepared pairs
# -------------------------
def load_pairs(path):
    data = np.load(path)

    return (
        data["pair_a"].astype(np.float32),
        data["pair_b"].astype(np.float32),
        data["labels"].astype(np.float32)
    )


a_train, b_train, y_train = load_pairs("prepared/train.npz")
a_val, b_val, y_val = load_pairs("prepared/val.npz")


# -------------------------
# 2. Create shared encoder
# -------------------------
def create_encoder():
    inputs = layers.Input(
        shape=(SEQUENCE_LENGTH, FEATURES)
    )

    x = layers.LSTM(128)(inputs)
    x = layers.Dense(64, activation="relu")(x)

    # Embedding vector
    outputs = layers.Dense(32)(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="shared_encoder"
    )


encoder = create_encoder()


# -------------------------
# 3. Siamese inputs
# -------------------------
input_a = layers.Input(
    shape=(SEQUENCE_LENGTH, FEATURES),
    name="sequence_a"
)

input_b = layers.Input(
    shape=(SEQUENCE_LENGTH, FEATURES),
    name="sequence_b"
)

embedding_a = encoder(input_a)
embedding_b = encoder(input_b)


# -------------------------
# 4. Euclidean distance
# -------------------------
def euclidean_distance(tensors):
    a, b = tensors

    squared_distance = tf.reduce_sum(
        tf.square(a - b),
        axis=1,
        keepdims=True
    )

    return tf.sqrt(
        tf.maximum(squared_distance, 1e-10)
    )


distance = layers.Lambda(
    euclidean_distance,
    name="distance"
)([embedding_a, embedding_b])


siamese = Model(
    inputs=[input_a, input_b],
    outputs=distance,
    name="siamese_network"
)


# -------------------------
# 5. Contrastive loss
# -------------------------
def contrastive_loss(y_true, y_pred):
    y_true = tf.cast(
        tf.reshape(y_true, (-1, 1)),
        tf.float32
    )

    margin = 1.0

    similar_loss = y_true * tf.square(y_pred)

    different_loss = (
        (1.0 - y_true)
        * tf.square(
            tf.maximum(margin - y_pred, 0.0)
        )
    )

    return tf.reduce_mean(
        similar_loss + different_loss
    )


# -------------------------
# 6. Compile
# -------------------------
siamese.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=contrastive_loss
)

siamese.summary()


# -------------------------
# 7. Train
# -------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_siamese.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = siamese.fit(
    [a_train, b_train],
    y_train,
    validation_data=(
        [a_val, b_val],
        y_val
    ),
    epochs=50,
    batch_size=16,
    callbacks=callbacks
)

# Save the shared encoder for later comparison
encoder.save("models/encoder.keras")

print("Training complete!")


# 4. Eveluate Model

In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

MODEL_PATH = "models/best_siamese.keras"

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "contrastive_loss": lambda y_true, y_pred: y_pred
    },
    compile=False
)


def load_pairs(path):
    data = np.load(path)

    return (
        data["pair_a"].astype(np.float32),
        data["pair_b"].astype(np.float32),
        data["labels"].astype(np.int32)
    )


# Get distances for each pair
def get_distances(path):
    a, b, y = load_pairs(path)

    distances = model.predict(
        [a, b],
        verbose=0
    ).reshape(-1)

    return distances, y


# -------------------------
# 1. Find threshold on VAL
# -------------------------
val_distances, y_val = get_distances(
    "prepared/val.npz"
)

candidates = np.linspace(
    val_distances.min(),
    val_distances.max(),
    500
)

best_threshold = None
best_accuracy = -1

for threshold in candidates:
    predictions = (
        val_distances <= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_val,
        predictions
    )

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = float(threshold)


Path("models/threshold.txt").write_text(
    str(best_threshold)
)

print("Chosen threshold:", best_threshold)
print("Validation pair accuracy:", best_accuracy)


# -------------------------
# 2. Evaluate on TEST once
# -------------------------
test_distances, y_test = get_distances(
    "prepared/test.npz"
)

test_predictions = (
    test_distances <= best_threshold
).astype(int)

print("\nTEST RESULTS")
print("Accuracy:", accuracy_score(
    y_test, test_predictions
))
print("Precision:", precision_score(
    y_test, test_predictions, zero_division=0
))
print("Recall:", recall_score(
    y_test, test_predictions, zero_division=0
))
print("F1:", f1_score(
    y_test, test_predictions, zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test, test_predictions
))


# 5. Test

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
from pathlib import Path
from collections import deque

# -------------------------
# 1. Configuration
# -------------------------
SEQUENCE_LENGTH = 30
FEATURES = 132
CAMERA_ID = 0

CORRECT_DIR = Path("dataset/correct")
MODEL_PATH = "models/best_siamese.keras"
THRESHOLD_PATH = Path("models/threshold.txt")

if not THRESHOLD_PATH.exists():
    raise FileNotFoundError(
        "Run evaluate_model.py first."
    )

threshold = float(
    THRESHOLD_PATH.read_text().strip()
)

if threshold <= 0:
    raise ValueError(
        "Threshold must be greater than zero."
    )

# Load trained Siamese model
model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# Load correct reference sequences
reference_files = sorted(
    CORRECT_DIR.glob("*.npy")
)

references = []

for file in reference_files:
    seq = np.load(file).astype(np.float32)

    if seq.shape == (SEQUENCE_LENGTH, FEATURES):
        references.append(seq)

if not references:
    raise RuntimeError(
        "No valid correct reference sequences found."
    )

references = np.stack(references)

print("Reference sequences:", len(references))
print("Threshold:", threshold)


# -------------------------
# 2. MediaPipe setup
# -------------------------
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils


def extract_landmarks(results):
    if results.pose_landmarks is None:
        return None

    landmarks = results.pose_landmarks.landmark

    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]

    center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2

    scale = np.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2
        + (left_shoulder.y - right_shoulder.y) ** 2
    )

    scale = max(scale, 1e-6)

    values = []

    for landmark in landmarks:
        x = (landmark.x - center_x) / scale
        y = (landmark.y - center_y) / scale
        z = landmark.z / scale
        visibility = landmark.visibility

        values.extend([x, y, z, visibility])

    return np.array(values, dtype=np.float32)


# -------------------------
# 3. Webcam
# -------------------------
cap = cv2.VideoCapture(CAMERA_ID)

if not cap.isOpened():
    raise RuntimeError("Cannot open webcam.")

sequence_buffer = deque(maxlen=SEQUENCE_LENGTH)

# Smooth the displayed score
score_buffer = deque(maxlen=5)

with mp_pose.Pose(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    while True:
        success, frame = cap.read()

        if not success:
            print("Cannot read webcam.")
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(
            frame, cv2.COLOR_BGR2RGB
        )

        results = pose.process(rgb)

        features = extract_landmarks(results)

        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

        if features is None:
            sequence_buffer.clear()
            score_buffer.clear()

            cv2.putText(
                frame,
                "Body not detected",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0, 0, 255), 2
            )

        else:
            sequence_buffer.append(features)

            cv2.putText(
                frame,
                f"Frames: {len(sequence_buffer)}/{SEQUENCE_LENGTH}",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2
            )

            if len(sequence_buffer) == SEQUENCE_LENGTH:
                live_sequence = np.array(
                    sequence_buffer,
                    dtype=np.float32
                )

                # Compare live sequence with each reference
                live_batch = np.repeat(
                    live_sequence[np.newaxis, ...],
                    len(references),
                    axis=0
                )

                distances = model.predict(
                    [
                        live_batch,
                        references
                    ],
                    verbose=0
                ).reshape(-1)

                # Use the mean of the 3 nearest references
                nearest_count = min(3, len(distances))

                nearest_distances = np.sort(
                    distances
                )[:nearest_count]

                distance = float(
                    np.mean(nearest_distances)
                )

                # Heuristic display score, NOT calibrated accuracy
                similarity = 100.0 * max(
                    0.0,
                    min(
                        1.0,
                        1.0 - distance / (2.0 * threshold)
                    )
                )

                score_buffer.append(similarity)

                displayed_score = float(
                    np.mean(score_buffer)
                )

                is_similar = distance <= threshold

                if is_similar:
                    status = "SIMILAR TO REFERENCE"
                    color = (0, 220, 0)
                else:
                    status = "CHECK YOUR POSE"
                    color = (0, 165, 255)

                cv2.putText(
                    frame,
                    f"Similarity: {displayed_score:.1f}%",
                    (20, 75),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, color, 2
                )

                cv2.putText(
                    frame,
                    status,
                    (20, 115),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, color, 2
                )

                cv2.putText(
                    frame,
                    f"Distance: {distance:.3f}",
                    (20, 150),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2
                )

        cv2.putText(
            frame,
            "Q: Quit",
            (20, frame.shape[0] - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (255, 255, 255), 2
        )

        cv2.imshow("SaluteAI - Realtime", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()
